<a href="https://colab.research.google.com/github/zharmo/ANN/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np

# ==========================================
# 1. GIVEN VALUES
# ==========================================

# Input image: 5 x 5
X = np.array([
    [1, 0, 2, 1, 0],
    [2, 1, 0, 1, 2],
    [0, 1, 2, 0, 1],
    [1, 2, 1, 0, 2],
    [2, 0, 1, 2, 1]
], dtype=float)

# Actual target
T = np.array([1.0, 0.0])

# Two convolution filters: 2 x 2 each
K = np.array([
    [[ 0.1, -0.1],
     [ 0.2,  0.1]],

    [[-0.2,  0.2],
     [ 0.1,  0.2]]
])

# Convolution biases
b_conv = np.array([0.05, 0.0])

# Dense weights: 8 inputs, 2 outputs
W_dense = np.array([
    [ 0.2, -0.1,  0.1,  0.2, -0.2,  0.1,  0.05,  0.15],
    [-0.1,  0.2,  0.15, -0.05, 0.1, -0.2,  0.2,   0.1]
])

b_dense = np.array([0.05, -0.05])

learning_rate = 0.1
lambda_reg = 0.01


# ==========================================
# 2. ACTIVATION FUNCTIONS
# ==========================================

def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def relu(z):
    return np.maximum(0, z)


# ==========================================
# 3. CONVOLUTION
# ==========================================

def convolution(X, K, b):

    filters, kh, kw = K.shape

    height, width = X.shape

    output = np.zeros((
        height - kh + 1,
        width - kw + 1,
        filters
    ))

    for f in range(filters):

        for i in range(output.shape[0]):

            for j in range(output.shape[1]):

                area = X[i:i+kh, j:j+kw]

                output[i, j, f] = (
                    np.sum(area * K[f]) + b[f]
                )

    return output


# ==========================================
# 4. MAX POOLING
# ==========================================

def max_pooling(A):

    height, width, channels = A.shape

    output = np.zeros((2, 2, channels))

    positions = np.zeros(
        (2, 2, channels, 2), dtype=int
    )

    for i in range(2):

        for j in range(2):

            for c in range(channels):

                area = A[
                    i*2:i*2+2,
                    j*2:j*2+2,
                    c
                ]

                index = np.unravel_index(
                    np.argmax(area), area.shape
                )

                output[i, j, c] = area[index]

                positions[i, j, c] = [
                    i*2 + index[0],
                    j*2 + index[1]
                ]

    return output, positions


# ==========================================
# 5. FORWARD PROPAGATION
# ==========================================

def forward(K, W_dense):

    # Step 1: Convolution
    Z_conv = convolution(X, K, b_conv)

    # Step 2: ReLU
    A_conv = relu(Z_conv)

    # Step 3: Max Pooling
    P, positions = max_pooling(A_conv)

    # Step 4: Flatten
    flat = P.reshape(-1)

    # Step 5: Dense layer
    Z_out = W_dense @ flat + b_dense

    # Step 6: Sigmoid
    Y_pred = sigmoid(Z_out)

    return (
        Z_conv,
        A_conv,
        P,
        positions,
        flat,
        Z_out,
        Y_pred
    )


# ==========================================
# 6. SQUARED ERROR LOSS
# ==========================================

def squared_loss(Y_pred):

    return 0.5 * np.sum((Y_pred - T) ** 2)


# ==========================================
# 7. L1 REGULARIZATION
# ==========================================

def L1_loss(K, W_dense):

    return lambda_reg * (
        np.sum(np.abs(K)) +
        np.sum(np.abs(W_dense))
    )


# ==========================================
# 8. L2 REGULARIZATION
# ==========================================

def L2_loss(K, W_dense):

    return (lambda_reg / 2) * (
        np.sum(K ** 2) +
        np.sum(W_dense ** 2)
    )


# ==========================================
# 9. INITIAL FORWARD PASS
# ==========================================

(
    Z_conv,
    A_conv,
    P,
    positions,
    flat,
    Z_out,
    Y_pred
) = forward(K, W_dense)

initial_loss = squared_loss(Y_pred)

print("\n========== FORWARD PASS ==========")

print("\nConvolution Z:")
print(Z_conv)

print("\nReLU A:")
print(A_conv)

print("\nMax Pooling:")
print(P)

print("\nFlatten:")
print(flat)

print("\nDense Z:")
print(Z_out)

print("\nPredicted Y:")
print(Y_pred)

print("\nSquared Error Loss:")
print(initial_loss)


# ==========================================
# 10. INITIAL REGULARIZATION
# ==========================================

initial_L1 = L1_loss(K, W_dense)
initial_L2 = L2_loss(K, W_dense)

print("\n========== REGULARIZATION ==========")

print("L1 penalty:", initial_L1)
print("L2 penalty:", initial_L2)

print("L1 total objective:", initial_loss + initial_L1)
print("L2 total objective:", initial_loss + initial_L2)


# ==========================================
# 11. BACKPROPAGATION
# ==========================================

# Output error
delta_out = (
    (Y_pred - T) *
    Y_pred *
    (1 - Y_pred)
)

# Dense weight gradient
grad_W_dense = np.outer(delta_out, flat)

# Backpropagate into flattened features
d_flat = W_dense.T @ delta_out

# Restore pooling dimensions
d_pool = d_flat.reshape(P.shape)

# Backpropagate through Max Pooling
d_A_conv = np.zeros_like(A_conv)

for i in range(2):

    for j in range(2):

        for c in range(2):

            r, col = positions[i, j, c]

            d_A_conv[r, col, c] += d_pool[i, j, c]


# Backpropagate through ReLU
d_Z_conv = d_A_conv * (Z_conv > 0)


# Convolution kernel gradients
grad_K = np.zeros_like(K)

for f in range(2):

    for i in range(4):

        for j in range(4):

            grad_K[f] += (
                d_Z_conv[i, j, f] *
                X[i:i+2, j:j+2]
            )


print("\n========== BACKPROPAGATION ==========")

print("Output error:")
print(delta_out)

print("\nDense gradients:")
print(grad_W_dense)

print("\nConvolution gradients:")
print(grad_K)


# ==========================================
# 12. UPDATE SELECTED WEIGHTS
# ==========================================

def update_weights(reg_type):

    new_K = K.copy()
    new_W = W_dense.copy()

    selected = [
        ("Conv filter 1 K[0,0]", new_K,
         grad_K, (0, 0, 0)),

        ("Dense output 1 W[0,0]", new_W,
         grad_W_dense, (0, 0)),

        ("Dense output 2 W[1,1]", new_W,
         grad_W_dense, (1, 1))
    ]

    print("\n==========", reg_type, "UPDATE ==========")

    for name, weight, gradient, index in selected:

        old = weight[index]

        data_gradient = gradient[index]

        if reg_type == "L1":

            reg_gradient = (
                lambda_reg * np.sign(old)
            )

        else:

            reg_gradient = lambda_reg * old

        total_gradient = (
            data_gradient + reg_gradient
        )

        updated = (
            old - learning_rate * total_gradient
        )

        weight[index] = updated

        print("\n", name)
        print("Old weight:", old)
        print("Data gradient:", data_gradient)
        print("Regularization gradient:", reg_gradient)
        print("Total gradient:", total_gradient)
        print("New weight:", updated)

    return new_K, new_W


# ==========================================
# 13. NEW FORWARD PASS AND LOSS
# ==========================================

def experiment(reg_type):

    new_K, new_W = update_weights(reg_type)

    (
        new_Z,
        new_A,
        new_P,
        new_positions,
        new_flat,
        new_Z_out,
        new_Y_pred
    ) = forward(new_K, new_W)

    new_loss = squared_loss(new_Y_pred)

    if reg_type == "L1":

        old_penalty = initial_L1
        new_penalty = L1_loss(new_K, new_W)

    else:

        old_penalty = initial_L2
        new_penalty = L2_loss(new_K, new_W)

    old_total = initial_loss + old_penalty
    new_total = new_loss + new_penalty

    print("\n========== NEW", reg_type, "FORWARD PASS ==========")

    print("\nNew convolution:")
    print(new_Z)

    print("\nNew ReLU:")
    print(new_A)

    print("\nNew Max Pooling:")
    print(new_P)

    print("\nNew Flatten:")
    print(new_flat)

    print("\nNew Dense Z:")
    print(new_Z_out)

    print("\nNew predicted output:")
    print(new_Y_pred)

    print("\nOld squared loss:", initial_loss)
    print("New squared loss:", new_loss)

    print("\nOld total objective:", old_total)
    print("New total objective:", new_total)

    print("\nTotal objective change:", new_total - old_total)

    if new_total < old_total:

        print("Objective decreased.")

    else:

        print("Objective increased.")


# ==========================================
# 14. RUN L1 AND L2 SEPARATELY
# ==========================================

experiment("L1")

experiment("L2")



========== FORWARD PASS ==========

Convolution Z:
[[[ 0.65  0.2 ]
  [ 0.05  0.5 ]
  [ 0.25  0.  ]
  [ 0.55  0.3 ]]

 [[ 0.25  0.  ]
  [ 0.55  0.3 ]
  [ 0.35  0.4 ]
  [ 0.05  0.4 ]]

 [[ 0.35  0.7 ]
  [ 0.45  0.6 ]
  [ 0.45 -0.3 ]
  [ 0.15  0.6 ]]

 [[ 0.35  0.4 ]
  [ 0.25  0.  ]
  [ 0.55  0.3 ]
  [ 0.35  0.8 ]]]

ReLU A:
[[[0.65 0.2 ]
  [0.05 0.5 ]
  [0.25 0.  ]
  [0.55 0.3 ]]

 [[0.25 0.  ]
  [0.55 0.3 ]
  [0.35 0.4 ]
  [0.05 0.4 ]]

 [[0.35 0.7 ]
  [0.45 0.6 ]
  [0.45 0.  ]
  [0.15 0.6 ]]

 [[0.35 0.4 ]
  [0.25 0.  ]
  [0.55 0.3 ]
  [0.35 0.8 ]]]

Max Pooling:
[[[0.65 0.5 ]
  [0.55 0.4 ]]

 [[0.45 0.7 ]
  [0.55 0.8 ]]]

Flatten:
[0.65 0.5  0.55 0.4  0.45 0.7  0.55 0.8 ]

Dense Z:
[0.3925 0.1425]

Predicted Y:
[0.59688438 0.53556484]

Squared Error Loss:
0.22466595008784312

========== REGULARIZATION ==========
L1 penalty: 0.034
L2 penalty: 0.0027500000000000007
L1 total objective: 0.2586659500878431
L2 total objective: 0.22741595008784313

========== BACKPROPAGATION ==========
Outp